In [ ]:
from google.colab import files
import numpy as np
import pandas as pd

# Загрузите data.bin и label.bin через диалог
uploaded = files.upload()

Saving data.bin to data.bin
Saving label.bin to label.bin


In [ ]:
import numpy as np
import pandas as pd

# 1. Читаем данные.
# ВАЖНО: Тебе нужно знать тип данных (dtype).
# Обычно это float32 для данных и int32 или int64 для меток.
try:
    # Загружаем данные
    data = np.fromfile('data.bin', dtype=np.float32)
    # Загружаем метки
    labels = np.fromfile('label.bin', dtype=np.int32)

    print(f"Загружено элементов данных: {len(data)}")
    print(f"Загружено меток: {len(labels)}")

    # 2. Если данные многомерные (например, картинки), их нужно "причесать" (reshape)
    # Если это просто плоский список, пропускаем этот шаг.
    # data = data.reshape(len(labels), -1)

    # 3. Создаем таблицу
    df = pd.DataFrame(data, columns=[f'feature_{i}' for i in range(data.shape[1])] if data.ndim > 1 else ['data'])
    df['label'] = labels

    # 4. Сохраняем в CSV
    output_filename = 'converted_data.csv'
    df.to_csv(output_filename, index=False)

    print(f"Готово! Файл сохранен как: {output_filename}")

except Exception as e:
    print(f"Произошла ошибка: {e}")
    print("Подсказка: проверь, правильно ли указан dtype (float32, float64, int32).")

Загружено элементов данных: 2035
Загружено меток: 7
Произошла ошибка: Length of values (7) does not match length of index (2035)
Подсказка: проверь, правильно ли указан dtype (float32, float64, int32).


In [ ]:
import os

data_size = os.path.getsize('data.bin')
label_size = os.path.getsize('label.bin')

print(f"Размер data.bin: {data_size} байт")
print(f"Размер label.bin: {label_size} байт")

Размер data.bin: 8142 байт
Размер label.bin: 29 байт


In [ ]:
print("--- Анализ label.bin ---")
with open('label.bin', 'rb') as f:
    lbl_bytes = f.read()

try:
    print(f"Текст внутри: {lbl_bytes.decode('utf-8')}")
except:
    print(f"Это не текст. Сырые байты (hex): {lbl_bytes.hex()}")

print("\n--- Анализ data.bin (первые 100 байт) ---")
with open('data.bin', 'rb') as f:
    data_bytes = f.read(100)

try:
    print(f"Текст внутри: {data_bytes.decode('utf-8')}")
except:
    print(f"Это не текст. Сырые байты (hex): {data_bytes.hex()}")

--- Анализ label.bin ---
Это не текст. Сырые байты (hex): 789c8b36d451a0041993824cc94226242223a2910185281600f0aa34bd

--- Анализ data.bin (первые 100 байт) ---
Это не текст. Сырые байты (hex): 789cd59c0b12e4368e6db7e205f829c43fb9968edeff36de39202565b5bb17309e890e573a532241e0e2e202ac7ffdabe57a8d55ef3ecacc33e5f6f75ff795e6a875963eeae8b59792fdf02e39a57ab7394a5eb357be99f3bce67de79256296d95c1677d


In [ ]:
import zlib
import numpy as np

def get_info(name, path):
    with open(path, 'rb') as f:
        raw = zlib.decompress(f.read())
    print(f"Файл: {name}")
    print(f"  - Распакованный размер: {len(raw)} байт")
    return raw

# Распаковываем
try:
    data_raw = get_info("DATA", "data.bin")
    label_raw = get_info("LABEL", "label.bin")

    print("\n--- Возможные варианты (делимость байтов) ---")
    for d_name, d_size in [("int8/uint8", 1), ("int16", 2), ("float32/int32", 4), ("float64/int64", 8)]:
        if len(data_raw) % d_size == 0:
            print(f"[DATA] Можно прочитать как {d_name} (получится {len(data_raw)//d_size} чисел)")
        if len(label_raw) % d_size == 0:
            print(f"[LABEL] Можно прочитать как {d_name} (получится {len(label_raw)//d_size} чисел)")

except Exception as e:
    print(f"Ошибка: {e}")

Файл: DATA
  - Распакованный размер: 17994 байт
Файл: LABEL
  - Распакованный размер: 318 байт

--- Возможные варианты (делимость байтов) ---
[DATA] Можно прочитать как int8/uint8 (получится 17994 чисел)
[LABEL] Можно прочитать как int8/uint8 (получится 318 чисел)
[DATA] Можно прочитать как int16 (получится 8997 чисел)
[LABEL] Можно прочитать как int16 (получится 159 чисел)


In [ ]:
import zlib
import numpy as np
import pandas as pd
import io

def smart_decode(file_path):
    with open(file_path, 'rb') as f:
        # 1. Распаковываем
        decompressed = zlib.decompress(f.read())

    # 2. Пытаемся прочитать как текст (вдруг там "1, 2, 3...")
    try:
        text = decompressed.decode('utf-8').strip()
        # Если в тексте есть запятые или пробелы, превращаем в список чисел
        if ',' in text:
            return np.array([float(x) for x in text.split(',') if x.strip()])
        else:
            return np.array([float(x) for x in text.split() if x.strip()])
    except:
        # 3. Если не текст, читаем как бинарные данные (2 байта на число - int16)
        # Так как 17994 и 318 делятся только на 2
        return np.frombuffer(decompressed, dtype=np.int16).copy()

try:
    data_array = smart_decode('data.bin')
    label_array = smart_decode('label.bin')

    print(f"Успешно извлечено!")
    print(f"Элементов в DATA: {len(data_array)}")
    print(f"Элементов в LABEL: {len(label_array)}")

    # 4. Пробуем собрать в таблицу
    # Если элементов в данных в N раз больше, чем в метках - это наши колонки
    if len(data_array) % len(label_array) == 0:
        cols = len(data_array) // len(label_array)
        df = pd.DataFrame(data_array.reshape(len(label_array), cols))
        df['label'] = label_array
        df.to_csv('result.csv', index=False)
        print(f"\n--- ГОТОВО! ---")
        print(f"Создан файл 'result.csv' ({len(label_array)} строк, {cols} признаков + метка)")
    else:
        print("\n[!] Внимание: Количество данных не кратно количеству меток.")
        print("Попробуем просто сохранить их в разные колонки, если это возможно.")
        # Если не делится, просто выведем первые значения для анализа
        print("Первые 5 меток:", label_array[:5])
        print("Первые 5 данных:", data_array[:5])

except Exception as e:
    print(f"Ошибка: {e}")

Успешно извлечено!
Элементов в DATA: 8997
Элементов в LABEL: 159

[!] Внимание: Количество данных не кратно количеству меток.
Попробуем просто сохранить их в разные колонки, если это возможно.
Первые 5 меток: [12635  8236 11313 12576  8236]
Первые 5 данных: [23387 12853 11828 14647 12340]


In [ ]:
import zlib
import re
import pandas as pd
import numpy as np

def extract_numbers_from_zlib(file_path):
    with open(file_path, 'rb') as f:
        # 1. Распаковываем
        decompressed_bytes = zlib.decompress(f.read())
        # 2. Декодируем байты в текст
        text_content = decompressed_bytes.decode('utf-8')

    # 3. Ищем все числа (целые и дробные) с помощью регулярных выражений
    # Это уберет скобки [ ], запятые и пробелы автоматически
    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", text_content)
    return np.array(numbers, dtype=float)

try:
    print("Распаковка и чтение текстовых данных...")
    data_values = extract_numbers_from_zlib('data.bin')
    label_values = extract_numbers_from_zlib('label.bin')

    print(f"Найдено чисел в DATA: {len(data_values)}")
    print(f"Найдено чисел в LABEL: {len(label_values)}")

    # Теперь проверяем кратность
    if len(data_values) % len(label_values) == 0:
        num_labels = len(label_values)
        num_features = len(data_values) // num_labels

        # Создаем таблицу
        df = pd.DataFrame(data_values.reshape(num_labels, num_features))
        df['label'] = label_values.astype(int) # Метки обычно целые

        # Сохраняем в CSV
        output_file = 'final_data_fixed.csv'
        df.to_csv(output_file, index=False)

        print(f"\n--- УСПЕХ! ---")
        print(f"Файл сохранен: {output_file}")
        print(f"Размер таблицы: {num_labels} строк и {num_features} признаков + 1 колонка меток.")
    else:
        print("\n[!] Ошибка: Количество чисел в DATA все еще не делится на количество меток.")
        print(f"Разница: {len(data_values)} / {len(label_values)} = {len(data_values)/len(label_values)}")
        print("Возможно, в конце файлов есть лишние данные или формат DATA отличается.")

except Exception as e:
    print(f"Произошла ошибка: {e}")

Распаковка и чтение текстовых данных...
Найдено чисел в DATA: 954
Найдено чисел в LABEL: 106

--- УСПЕХ! ---
Файл сохранен: final_data_fixed.csv
Размер таблицы: 106 строк и 9 признаков + 1 колонка меток.


In [ ]:
df = pd.read_csv('final_data_fixed.csv')
display(df.head()) # Покажет первые 5 строк

,0,1,2,3,4,5,6,7,8,label
0,524.794067,0.187448,0.032114,228.800232,6843.598633,29.910803,60.204880,220.737213,556.828308,1
1,330.000000,0.226893,0.265290,121.154198,3163.239502,26.109201,69.717361,99.084961,400.225769,1
2,551.879272,0.232478,0.063530,264.804932,11888.391602,44.894901,77.793297,253.785294,656.769470,1
3,380.000000,0.240855,0.286234,137.640106,5402.171387,39.248524,88.758446,105.198570,493.701813,1
4,362.831268,0.200713,0.244346,124.912560,3290.462402,26.342127,69.389389,103.866554,424.796509,1
